# Análisis de Datos · Semana 10
## Funciones definidas por el usuario

**TIA502 · Facultad de Empresariales · Profesor David Escobar-Castillejos**

Ya usas `PROMEDIO` en una hoja sin saber cómo suma ni cómo divide. Hoy aprendes a fabricar el
tuyo.

El argumento que convence no es "reutilizar código". Es que **una función se puede probar sola**,
y una fórmula pegada en trescientas celdas no.

Al terminar este cuaderno vas a poder:

1. Explicar qué resuelve una función, más allá de ahorrar líneas.
2. Definir una función con `def`, con nombre, parámetros y cuerpo.
3. Distinguir parámetro de argumento.
4. Devolver un valor con `return`, y decir en qué se diferencia de imprimirlo.
5. Reconocer el ámbito de un nombre y por qué lo de adentro no sale.

### Cómo se usa este cuaderno

Ejecuta las celdas en orden. Cinco fallan a propósito y llevan un comentario que lo dice.

El caso de aquí a la semana 13 es de finanzas: el pago mensual de un crédito.

---
# Bloque 1 · Por qué existen las funciones

No para escribir menos. Para tener **un solo lugar** donde el cálculo pueda estar bien o estar
mal.

Así se ve el mismo cálculo repetido a mano, para dos créditos:

In [ ]:
# Crédito A
i = 0.18 / 12
pago_a = 250000 * (i * (1 + i) ** 36) / ((1 + i) ** 36 - 1)

# Crédito B
i = 0.24 / 12
pago_b = 120000 * (i * (1 + i) ** 24) / ((1 + i) ** 24 - 1)

print(f"A: {pago_a:,.2f}")
print(f"B: {pago_b:,.2f}")

Funciona, y tiene dos problemas que no se ven al leerlo.

La fórmula está escrita dos veces, así que si está mal hay que encontrar y arreglar cada copia.
Y la variable `i` se reusó: la segunda línea pisó a la primera, y si alguien mueve el orden de
los bloques el resultado cambia sin avisar.

La misma cuenta, empaquetada:

In [ ]:
def pago_mensual(capital, tasa_anual, meses):
    i = tasa_anual / 12
    factor = (1 + i) ** meses
    return capital * (i * factor) / (factor - 1)


pago_a = pago_mensual(250000, 0.18, 36)
pago_b = pago_mensual(120000, 0.24, 24)

print(f"A: {pago_a:,.2f}")
print(f"B: {pago_b:,.2f}")

Los mismos dos números. La diferencia es qué pasa cuando la fórmula está mal: arriba hay que
arreglar cada copia, abajo se arregla una vez y las dos llamadas quedan correctas.

## El argumento de verdad

Una función **se puede probar sola**. Una fórmula pegada en trescientas celdas solo se puede
revisar celda por celda, y nadie lo hace.

Probar significa esto: le das entradas cuya respuesta ya conoces y compruebas que la devuelve.

In [ ]:
# Un crédito de 12,000 al 0.0000001 % anual a 12 meses debería pagar casi 1,000 al mes,
# porque casi no hay intereses.
print("Sin intereses prácticamente:", round(pago_mensual(12000, 0.0000001, 12), 2))

# Y el pago siempre tiene que ser mayor que capital entre meses, porque hay intereses.
print("Pago:", round(pago_mensual(250000, 0.18, 36), 2))
print("Capital entre meses:", round(250000 / 36, 2))
print("¿El pago es mayor?", pago_mensual(250000, 0.18, 36) > 250000 / 36)

Esas tres comprobaciones caben en una celda y se pueden volver a correr cada vez que alguien
toque la fórmula. Eso es lo que la hoja de cálculo no te da.

---
# Bloque 2 · Cómo se escribe una función

Cinco partes, y cada una tiene una regla que no se negocia.

| Parte | Qué es | En el ejemplo |
|---|---|---|
| `def` | La palabra que la declara | `def` |
| Nombre | Cómo se le llama después | `pago_mensual` |
| Parámetros | Los huecos que hay que llenar | `capital, tasa_anual, meses` |
| Cuerpo | El cálculo, con sangría | Las tres líneas de adentro |
| `return` | Lo que entrega al terminar | El pago calculado |

In [ ]:
def pago_mensual(capital, tasa_anual, meses):
    """Calcula el pago fijo mensual de un crédito.

    capital     lo que se presta, en pesos
    tasa_anual  la tasa nominal anual, como decimal: 0.18 es 18 %
    meses       el plazo
    """
    i = tasa_anual / 12
    factor = (1 + i) ** meses

    return capital * (i * factor) / (factor - 1)


pago = pago_mensual(250000, 0.18, 36)
print(f"Pago mensual: ${pago:,.2f}")

**`def` y el nombre.** El nombre dice qué devuelve, no qué hace por dentro. `pago_mensual`, no
`calcular_cosas`.

**Los parámetros.** Los tres huecos que la función necesita. Al llamarla se llenan en ese mismo
orden.

**El docstring**, esa cadena entre comillas triples. Explica para qué sirve, y lo lee quien la
use sin abrir el cuerpo. Python lo guarda y se puede consultar.

In [ ]:
print(pago_mensual.__doc__)

In [ ]:
help(pago_mensual)

**`return`** devuelve el resultado y termina la función ahí mismo. Lo que venga después no corre.

In [ ]:
def con_codigo_muerto(capital):
    return capital * 1.16
    print("Esta línea nunca se ejecuta")


print(con_codigo_muerto(1000))
print("Y no apareció ningún mensaje de adentro.")

## Parámetro no es lo mismo que argumento

Se confunden todo el tiempo y la distinción es sencilla.

**El parámetro** es el hueco que dejas al definirla: `capital`, `tasa_anual`, `meses`.

**El argumento** es el valor que llega al llamarla: `250000`, `0.18`, `36`.

El parámetro vive en la definición y el argumento en la llamada. Uno es la etiqueta del cajón,
el otro es lo que metes.

In [ ]:
def describir_credito(capital, tasa_anual, meses):
    """Los parámetros son capital, tasa_anual y meses."""
    return f"{capital:,} pesos al {tasa_anual:.0%} a {meses} meses"


# Aquí los argumentos son 250000, 0.18 y 36.
print(describir_credito(250000, 0.18, 36))

# Y se pueden pasar por nombre, en cualquier orden.
print(describir_credito(meses=24, capital=120000, tasa_anual=0.24))

Pasarlos por nombre cuesta más letras y quita toda duda. Con tres números seguidos, nadie que
lea `pago_mensual(250000, 0.18, 36)` puede jurar cuál es cuál sin ir a ver la definición.

In [ ]:
# FALLA A PROPÓSITO, y de dos formas distintas según cuáles dos intercambies.
import itertools

for orden in itertools.permutations([250000, 0.18, 36]):
    etiqueta = f"{orden[0]:>10,} {orden[1]:>10,} {orden[2]:>10,}"
    try:
        print(f"{etiqueta} -> {pago_mensual(*orden):>18,.2f}")
    except OverflowError as e:
        print(f"{etiqueta} -> OverflowError: {e}")

Seis órdenes posibles y tres desenlaces.

Uno es correcto, 9,038.10. Dos truenan con `OverflowError`, porque elevar 1.015 a la
doscientos cincuenta mil se sale de lo que cabe en un decimal. Y **tres devuelven un número sin
protestar.**

El peor de los tres es `pago_mensual(0.18, 250000, 36)`, que da 3,750.00. Es un pago mensual
perfectamente creíble para un crédito, y está calculado con el capital en el lugar de la tasa.

Los que truenan te avisan. El de 3,750 se va al reporte.

In [ ]:
# Por eso conviene pasarlos por nombre cuando son varios y del mismo tipo.
print(pago_mensual(capital=250000, tasa_anual=0.18, meses=36))

# Y así, ni siquiera importa el orden en que los escribas.
print(pago_mensual(meses=36, capital=250000, tasa_anual=0.18))

## Reutilizada

In [ ]:
creditos = [
    (250000, 0.18, 36),
    (120000, 0.24, 24),
    (80000, 0.15, 12),
]

print(f"{'Capital':>10}{'Tasa':>7}{'Meses':>7}{'Pago':>12}")
print("-" * 36)
for capital, tasa, meses in creditos:
    print(f"{capital:>10,}{tasa:>7.0%}{meses:>7}{pago_mensual(capital, tasa, meses):>12,.2f}")

Tres créditos, una sola fórmula. Agrega un cuarto a la lista y el ciclo no se toca.

## Devolver no es imprimir

Este es el error que más aparece en la primera entrega con funciones.

In [ ]:
# FALLA A PROPÓSITO. La función imprime en lugar de devolver.
def pago_que_imprime(capital, tasa_anual, meses):
    i = tasa_anual / 12
    factor = (1 + i) ** meses
    print(capital * (i * factor) / (factor - 1))


total = pago_que_imprime(250000, 0.18, 36)

print("Lo que quedó en total:", total)
print("Su tipo:", type(total))

El número apareció en pantalla y `total` quedó en `None`. Una función que solo imprime es un
callejón sin salida: no se puede sumar, ni guardar, ni graficar.

Y el `None` no truena ahí. Truena tres líneas después.

In [ ]:
# FALLA A PROPÓSITO. El None de arriba, usado como si fuera un número.
try:
    print(total * 36)
except TypeError as e:
    print("TypeError:", e)

Ese es el patrón que hay que reconocer: **`TypeError` con `NoneType` casi siempre significa que
a una función le falta el `return`.**

La regla: la función devuelve, y quien la llama decide si imprime.

In [ ]:
def pago_que_devuelve(capital, tasa_anual, meses):
    i = tasa_anual / 12
    factor = (1 + i) ** meses
    return capital * (i * factor) / (factor - 1)


total = pago_que_devuelve(250000, 0.18, 36)

print("El pago:", round(total, 2))
print("A 36 meses:", round(total * 36, 2))
print("Intereses:", round(total * 36 - 250000, 2))

El mismo cálculo, y ahora el resultado sirve para tres cosas más.

## Una función que usa otra

Con la de arriba ya se puede escribir la que reporta el crédito completo, sin volver a escribir
la fórmula.

In [ ]:
def resumen_credito(capital, tasa_anual, meses):
    """Devuelve el pago mensual, el total pagado y los intereses."""
    pago = pago_mensual(capital, tasa_anual, meses)
    total = pago * meses
    return pago, total, total - capital


pago, total, intereses = resumen_credito(250000, 0.18, 36)

print(f"Pago mensual: {pago:>12,.2f}")
print(f"Total pagado: {total:>12,.2f}")
print(f"Intereses:    {intereses:>12,.2f}")
print(f"Los intereses son el {intereses / capital if False else intereses / 250000:.1%} del capital")

`return pago, total, total - capital` devuelve tres cosas de golpe, empaquetadas en una tupla.
La línea que la recibe las desempaca en tres nombres.

Y fíjate en lo que **no** hay: `resumen_credito` no vuelve a escribir la fórmula del pago. La
pide. Si mañana descubres que la fórmula estaba mal, la arreglas en un lugar y las dos funciones
quedan correctas.

---
# Bloque 3 · Dónde vive cada nombre

Lo que se declara dentro de una función existe solo mientras esa función corre.

In [ ]:
def calcular(capital):
    comision = capital * 0.02
    return capital + comision


print(calcular(250000))

# FALLA A PROPÓSITO. comision nació y murió dentro de la función.
try:
    print(comision)
except NameError as e:
    print("NameError:", e)

Las variables que nacen dentro de una función son suyas. Se crean cuando la llamas y desaparecen
cuando termina.

Eso no es una limitación, es la garantía de que una función no puede romper el resto del
programa por accidente. Y trae una consecuencia buena: **puedes reusar el mismo nombre sin
miedo.**

In [ ]:
def una(x):
    factor = x * 2
    return factor


def otra(x):
    factor = x * 100      # el mismo nombre, y no se estorban
    return factor


print(una(5), otra(5))

El `factor` de una función y el `factor` de la otra son dos variables distintas que nunca se van
a ver.

## Lo que sí se ve desde adentro

Una función puede **leer** un nombre de afuera. Eso funciona y casi siempre es mala idea.

In [ ]:
IVA = 0.16                    # una constante del programa

def con_iva(monto):
    return monto * (1 + IVA)   # lee IVA de afuera


print(con_iva(1000))

Funciona porque `IVA` está definido cuando la función corre. El problema aparece al mover la
función a otro archivo: se lleva su cuerpo y deja `IVA` atrás.

In [ ]:
# FALLA A PROPÓSITO. La misma función, sin la constante que daba por hecha.
def con_impuesto(monto):
    return monto * (1 + TASA_QUE_NO_EXISTE)


try:
    print(con_impuesto(1000))
except NameError as e:
    print("NameError:", e)

La versión que sobrevive a la mudanza recibe todo lo que necesita.

In [ ]:
def con_iva_portatil(monto, iva=0.16):
    """Todo lo que necesita entra por la puerta."""
    return monto * (1 + iva)


print(con_iva_portatil(1000))
print(con_iva_portatil(1000, 0.08))      # zona fronteriza

Ese `iva=0.16` es un argumento por omisión, y es el tema de la semana que entra. Va aquí para
que veas que la solución existe.

## Modificar desde adentro

Leer de afuera funciona. Asignar, no.

In [ ]:
contador = 0

def sumar_uno():
    contador = contador_local = 1     # esto crea una variable NUEVA, local
    return contador


print("Devuelve:", sumar_uno())
print("Y la de afuera sigue en:", contador)

La de adentro y la de afuera se llaman igual y no son la misma. La asignación creó una local que
murió al terminar la función.

Existe una palabra para forzar lo contrario, `global`, y este curso no la usa. Una función que
modifica variables de afuera es exactamente la que no se puede probar sola, que es todo lo que
estamos tratando de evitar.

## Cuatro errores de la primera función

**Olvidar el `return`.** La función corre, calcula bien y devuelve `None`. El error aparece
líneas después.

**Predice antes de correr.** ¿Qué imprime este programa?

- **A.** 42, porque multiplica por dos.
- **B.** `None`, porque a la función le falta el `return`.
- **C.** 21, porque `n` no cambió.
- **D.** Un error, porque la función no hace nada.

In [ ]:
def duplicar(n):
    n * 2


resultado = duplicar(21)
print(resultado)

La respuesta es **B**. La multiplicación ocurrió, el resultado se calculó, y nadie lo devolvió.
Python entrega `None` cuando una función termina sin `return`.

**Imprimir en lugar de devolver.** Ya lo viste.

**Llamarla antes de definirla.** Python lee de arriba abajo.

In [ ]:
# FALLA A PROPÓSITO. La llamada va antes que la definición.
try:
    print(todavia_no_existe(10))
except NameError as e:
    print("NameError:", e)


def todavia_no_existe(x):
    return x * 2

En un cuaderno esto muerde distinto: si corres la celda de la definición y **después** una de más
arriba, la función sí existe, porque el estado es el de la última celda que ejecutaste, no el del
orden en pantalla.

Es la misma advertencia de la semana 3, y aquí es donde empieza a costar caro.

**Confiar en variables de fuera.** Ya lo viste con `IVA`.

---
# Ejercicios

Las soluciones están hasta abajo del cuaderno.

## Escribir funciones

### Ejercicio 1 · Tres funciones cortas

Escribe tres funciones con docstring, cada una con dos parámetros y un `return`:

1. `porcentaje(parte, total)` que devuelva qué fracción es la parte del total.
2. `variacion(actual, anterior)` que devuelva el cambio porcentual.
3. `con_descuento(precio, porcentaje_descuento)` que devuelva el precio final.

Pruébalas con dos casos cada una.

### Ejercicio 2 · El caso límite de cada una

Para las tres del ejercicio anterior, encuentra el valor de entrada que las rompe y compruébalo.

Pista: piensa qué pasa con un total de cero, con un anterior de cero, y con un descuento del
120 %.

### Ejercicio 3 · Devolver varias cosas

Escribe `estadisticas(numeros)` que reciba una lista y devuelva cuatro valores: la suma, el
promedio, el mayor y el menor. Desempácalos en cuatro nombres al llamarla.

### Ejercicio 4 · Una que usa a otra

Escribe `tabla_amortizacion(capital, tasa_anual, meses)` que use `pago_mensual` y devuelva una
lista de tuplas, una por mes, con el número de mes, el interés de ese mes, el abono a capital y
el saldo restante.

El interés de cada mes es el saldo por la tasa mensual. El abono a capital es el pago menos el
interés.

Comprueba que el saldo del último mes queda prácticamente en cero.

## Ámbito y errores

### Ejercicio 5 · El `None` que revienta después

Escribe a propósito una función sin `return`, guárdala en una variable, y después provoca los
tres errores distintos que ese `None` puede causar: sumarlo, indexarlo y llamar un método suyo.

Anota el mensaje de cada uno.

### Ejercicio 6 · Portátil o no

Esta función depende de algo que no recibe:

```python
COMISION = 0.02

def total_con_comision(monto):
    return monto * (1 + COMISION)
```

Reescríbela para que sea portátil, y demuestra que la primera se rompe borrando la constante y
la segunda no.

### Ejercicio 7 · El nombre repetido

Escribe dos funciones que usen internamente una variable llamada `total` para cosas distintas, y
además una variable `total` fuera de las dos. Imprime las tres y comprueba que ninguna estorba a
las otras.

## Con tu área

### Ejercicio 8 · Empaqueta un cálculo tuyo

Escribe una función que resuelva un cálculo real de tu carrera, con al menos dos parámetros y un
`return`. Escribe su docstring y pruébala con tres casos, incluido uno en el límite.

La función no puede imprimir nada. Solo recibe y devuelve.

La prueba: bórrale una línea al cuerpo. Si las tres pruebas siguen pasando, tus casos no probaban
nada.

---
## Tres ideas para llevarse

**Una función se puede probar sola.** Ese es el argumento de verdad. Ahorrar líneas es apenas el
efecto secundario más visible.

**Devolver no es imprimir.** Una función que solo imprime entrega `None`, y ese `None` revienta
tres líneas más abajo con un `TypeError` que no menciona a la función.

**Lo de adentro se queda adentro.** Y por eso puedes reusar los mismos nombres en dos funciones
sin que se estorben, y por eso una función bien escrita no puede romper el resto del programa.

La siguiente sesión son argumentos por omisión, funciones predefinidas y los módulos que ya
vienen con Python.

---
# Soluciones

### Ejercicio 1

```python
def porcentaje(parte, total):
    """Qué fracción del total representa la parte, como decimal."""
    return parte / total


def variacion(actual, anterior):
    """El cambio porcentual respecto al periodo anterior, como decimal."""
    return (actual - anterior) / anterior


def con_descuento(precio, porcentaje_descuento):
    """El precio final después de aplicar un descuento dado como decimal."""
    return precio * (1 - porcentaje_descuento)


print(f"{porcentaje(5074, 148230):.2%}")
print(f"{variacion(148230, 96400):+.1%}")
print(f"{con_descuento(8990, 0.15):,.2f}")
```

Nota el `+` en `{:+.1%}`: fuerza el signo, así que un crecimiento se lee `+53.8%` y una caída
`-12.0%`. En un reporte de variación eso vale más que el número solo.

### Ejercicio 2

```python
for f, args, etiqueta in [(porcentaje, (5074, 0), "total de cero"),
                          (variacion, (100, 0), "anterior de cero"),
                          (con_descuento, (8990, 1.2), "descuento del 120 %")]:
    try:
        print(f"{etiqueta:<22} -> {f(*args)}")
    except ZeroDivisionError as e:
        print(f"{etiqueta:<22} -> ZeroDivisionError: {e}")
```

Las dos primeras truenan y la tercera no: devuelve un precio negativo, `-1798.0`.

Esa es la peligrosa. Un error que truena te avisa; uno que devuelve un precio negativo se va a la
factura. Si la función va a usarse de verdad, ahí hace falta una validación.

### Ejercicio 3

```python
def estadisticas(numeros):
    """Devuelve suma, promedio, mayor y menor de una lista de números."""
    return sum(numeros), sum(numeros) / len(numeros), max(numeros), min(numeros)


suma, promedio, mayor, menor = estadisticas([23200, 42800, 82700, 24500, 24500])

print(f"Suma:     {suma:>10,}")
print(f"Promedio: {promedio:>10,.2f}")
print(f"Mayor:    {mayor:>10,}")
print(f"Menor:    {menor:>10,}")
```

Con una lista vacía truena en la división. Vale la pena decidir si eso está bien: para una
función de estadísticas, probablemente sí, porque el promedio de nada no existe.

### Ejercicio 4

```python
def tabla_amortizacion(capital, tasa_anual, meses):
    """Una tupla por mes: número, interés, abono a capital y saldo."""
    pago = pago_mensual(capital, tasa_anual, meses)
    i = tasa_anual / 12
    saldo = capital
    filas = []
    for mes in range(1, meses + 1):
        interes = saldo * i
        abono = pago - interes
        saldo -= abono
        filas.append((mes, interes, abono, saldo))
    return filas


tabla = tabla_amortizacion(250000, 0.18, 36)

print(f"{'Mes':>4}{'Interés':>12}{'Abono':>12}{'Saldo':>14}")
for mes, interes, abono, saldo in tabla[:3]:
    print(f"{mes:>4}{interes:>12,.2f}{abono:>12,.2f}{saldo:>14,.2f}")
print("  ...")
for mes, interes, abono, saldo in tabla[-2:]:
    print(f"{mes:>4}{interes:>12,.2f}{abono:>12,.2f}{saldo:>14,.2f}")

print(f"\nSaldo final: {tabla[-1][3]:.10f}")
```

El saldo final queda en algo como `0.0000000005`, no en cero exacto. Es el mismo redondeo binario
de la semana 4: cada mes arrastra una fracción de centavo.

En un sistema real el último pago se ajusta para cerrar exacto, y eso es una decisión de negocio,
no un error de la fórmula.

### Ejercicio 5

```python
def sin_return(x):
    x * 2


vacio = sin_return(21)

for etiqueta, accion in [("sumarlo", lambda: vacio + 1),
                         ("indexarlo", lambda: vacio[0]),
                         ("llamar un método", lambda: vacio.upper())]:
    try:
        accion()
    except TypeError as e:
        print(f"{etiqueta:<18} TypeError: {e}")
    except AttributeError as e:
        print(f"{etiqueta:<18} AttributeError: {e}")
```

Los tres mencionan `NoneType` y ninguno menciona `sin_return`. Por eso reconocer la palabra
`NoneType` en un mensaje vale tanto: es la pista de que el problema está en una función que no
devolvió, y no en la línea que truena.

### Ejercicio 6

```python
def total_con_comision_portatil(monto, comision=0.02):
    """Todo lo que necesita entra como parámetro."""
    return monto * (1 + comision)


print(total_con_comision_portatil(1000))
print(total_con_comision_portatil(1000, 0.05))

def total_con_comision(monto):
    return monto * (1 + COMISION_QUE_NO_DEFINI)

try:
    total_con_comision(1000)
except NameError as e:
    print("La primera versión:", e)
```

La portátil funciona sola y además ganó flexibilidad: la comisión ahora se puede cambiar por
llamada sin tocar la función. Ese es el patrón, y la semana 11 lo formaliza.

### Ejercicio 7

```python
total = "el de afuera"

def suma_precios(precios):
    total = sum(precios)
    return total

def cuenta_items(items):
    total = len(items)
    return total

print(suma_precios([100, 200, 300]))
print(cuenta_items(["a", "b", "c", "d"]))
print(total)
```

Sale 600, 4 y `el de afuera`. Tres variables con el mismo nombre y ninguna sabe de las otras.

Que el de afuera sea texto y los de adentro números es a propósito: si se estorbaran, algo habría
tronado.

### Ejercicio 8

No hay solución publicada porque el cálculo es distinto para cada quien. Se califica sobre cuatro
cosas: que tenga docstring, que no imprima nada, que las tres pruebas incluyan un caso límite, y
que borrarle una línea al cuerpo haga fallar al menos una prueba.

Esa última es la que de verdad mide. Una prueba que sigue pasando con la función rota no estaba
probando nada.